# 00 - Variables del Workshop BNCR

Define las variables globales para todos los notebooks del taller.

**Catálogo:** `BNS` (Banco Nacional de Costa Rica)


In [ ]:
catalog_name = "BNS"
schema_raw = "raw"
schema_bronze = "bronze"
schema_silver = "silver"
schema_gold = "gold"
volume = "transacciones"

vol_path = f"/Volumes/{catalog_name}/{schema_raw}/{volume}"
repo_url = "https://github.com/Ricojacob01/Latam_resources_spanish"
repo_path = "Data Engineering"

print(f"Catálogo  : {catalog_name}")
print(f"Volumen   : {vol_path}")
print(f"Repositorio: {repo_url}/{repo_path}")


In [ ]:
%pip install gitpython -q
dbutils.library.restartPython()


In [ ]:
def carga_datos(carga):
    """Carga archivos initial o incremental desde GitHub al Volume UC."""
    import git, os, shutil, tempfile

    if carga not in ("initial", "incremental"):
        raise ValueError("carga debe ser 'initial' o 'incremental'")

    dbutils.fs.mkdirs(vol_path)

    with tempfile.TemporaryDirectory() as tmp_dir:
        repo = git.Repo.clone_from(repo_url, tmp_dir, depth=1, no_checkout=True)
        repo.git.checkout('HEAD', '--', f'{repo_path}/Files/' + carga)
        src = os.path.join(tmp_dir, repo_path, 'Files', carga)
        for item in os.listdir(src):
            item_path = os.path.join(src, item)
            dst = os.path.join(vol_path, item)
            if os.path.isdir(item_path):
                if os.path.exists(dst):
                    shutil.rmtree(dst)
                shutil.copytree(item_path, dst)
            else:
                shutil.copy2(item_path, dst)

    print(f"Carga '{carga}' completada en {vol_path}")
    display(dbutils.fs.ls(vol_path))
